In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from urllib.request import urlopen

# Load tinyshakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urlopen(url).read().decode("utf-8")
text = text[:60000]  # keep runtime and memory manageable

print("Text length:", len(text))
print(text[:300])

Text length: 60000
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [2]:
# Create character mappings and training sequences
chars = sorted(list(set(text)))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}

maxlen = 40
step = 3
sentences = []
next_chars = []

for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i:i + maxlen])
    next_chars.append(text[i + maxlen])

print("Unique chars:", len(chars))
print("Number of sequences:", len(sentences))

Unique chars: 59
Number of sequences: 19987


In [3]:
# Vectorize inputs (binary matrices) and outputs (one-hot vectors)
X = np.zeros((len(sentences), maxlen, len(chars)), dtype=np.float32)
y = np.zeros((len(sentences), len(chars)), dtype=np.float32)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_to_idx[char]] = 1.0
    y[i, char_to_idx[next_chars[i]]] = 1.0

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (19987, 40, 59)
y shape: (19987, 59)


In [4]:
# Build and train a simple RNN model
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        return self.fc(out)

model = SimpleRNN(input_size=len(chars), hidden_size=128, output_size=len(chars))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

epochs = 10
for epoch in range(epochs):
    total_loss = 0.0
    for x_batch, y_batch in dataloader:
        target_idx = torch.argmax(y_batch, dim=1)
        logits = model(x_batch)
        loss = criterion(logits, target_idx)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs} - Loss: {total_loss / len(dataloader):.4f}")

Epoch 1/10 - Loss: 2.9972
Epoch 2/10 - Loss: 2.3719
Epoch 3/10 - Loss: 2.2155
Epoch 4/10 - Loss: 2.1331
Epoch 5/10 - Loss: 2.0741
Epoch 6/10 - Loss: 2.0214
Epoch 7/10 - Loss: 1.9674
Epoch 8/10 - Loss: 1.9230
Epoch 9/10 - Loss: 1.8857
Epoch 10/10 - Loss: 1.8422


In [5]:
# Generate text with a trained model for a specified word count
def generate_text(model, seed_text, word_count=100, temperature=1.0):
    model.eval()
    generated = seed_text

    while len(generated.split()) < word_count:
        current = generated[-maxlen:]
        x_pred = np.zeros((1, maxlen, len(chars)), dtype=np.float32)

        for t, ch in enumerate(current):
            if ch in char_to_idx:
                x_pred[0, t, char_to_idx[ch]] = 1.0

        with torch.no_grad():
            logits = model(torch.tensor(x_pred))
            probs = torch.softmax(logits / temperature, dim=-1).squeeze(0)
            next_idx = torch.multinomial(probs, num_samples=1).item()

        generated += idx_to_char[next_idx]

    return " ".join(generated.split()[:word_count])

seed = "ROMEO: "
sample_text = generate_text(model, seed_text=seed, word_count=100, temperature=0.8)
print(sample_text)
print("\nWord count:", len(sample_text.split()))

ROMEO: 'ioyusitaei r' nleosa ioor nrt ers beck bo nom shurd I preach. SIRGTLA: Shath hend-will. Seling arnous. Hiolins, when the VingU: CoRIOLANUS: Sin the harhs of ald and my is of the fir the poost fis keromt O tius, whire fiome: the fipectoruson The gros the blaust mecis the blather them and molt at oue the for you gres are ho datis of has well compiond the breash with the first They serams, and ancess all wo bughed; se the I nor shampes on wains man he pathing bremand offirs; far is char for wein o thele. VIlUFID: Y

Word count: 100
